# Visualizing IDC data in 3D with trame-slicer

<a href="https://colab.research.google.com/github/ImagingDataCommons/IDC-Tutorials/blob/master/notebooks/advanced_topics/trame_slicer_visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Updated: Jun 2026

This notebook shows how to view [NCI Imaging Data Commons (IDC)](https://imaging.datacommons.cancer.gov) images **interactively, in 3D, inside a Colab cell** using [**trame-slicer**](https://github.com/KitwareMedical/trame-slicer).

`trame-slicer` brings the [3D Slicer](https://www.slicer.org) rendering engine (MRML scene, multi‑planar reformat, volume rendering, segmentation display) to the browser through [trame](https://kitware.github.io/trame/). Rendering happens server‑side on the Colab VM and is streamed to the notebook output, so you get a real medical‑image viewer that understands orientation, windowing, spacing, and can overlay segmentations — without installing any desktop software.

> **Background.** This tutorial was put together in response to [trame-slicer issue #81](https://github.com/KitwareMedical/trame-slicer/issues/81), where the IDC team described the need for a robust, notebook‑native 3D viewer for medical images to replace fragile alternatives such as `itkwidgets`. It also addresses the request in that thread to demonstrate viewing a DICOM image together with a DICOM segmentation.

**What you will do**
1. Set up a headless OpenGL environment and install the packages.
2. Find a CT scan and a matching segmentation in IDC with `idc-index`.
3. Download the DICOM data.
4. Build a small, reusable trame-slicer viewer.
5. **Demo 1** — explore a CT volume with a 4‑up (Axial / Coronal / Sagittal / 3D) layout and interactive volume rendering.
6. **Demo 2** — overlay a whole‑body segmentation on the CT.

**Runtime note.** A standard **CPU** runtime is sufficient (rendering uses software OpenGL). The first run installs the 3D Slicer Python core, which can take a few minutes.

## 1. Set up the environment

The cell below installs the system OpenGL libraries that let VTK render **off‑screen** on a headless Colab VM (Mesa + OSMesa software rendering — the same setup `trame-slicer` uses in its own CI), then installs the Python packages:

- `trame-slicer[standalone]` — the viewer plus the bundled 3D Slicer Python core (`slicer-core`);
- `idc-index` — querying and downloading IDC data;
- `dcmqi` — converting a DICOM Segmentation object to a labelmap that the viewer can load.

In [ ]:
%%capture --no-stderr
# System OpenGL / off-screen rendering libraries (headless Colab VM).
!apt-get -qq update
!apt-get -qq install -y libgl1-mesa-glx libglu1-mesa libosmesa6 > /dev/null

# Python packages. trame-slicer[standalone] also installs the 3D Slicer core.
!pip install -q "trame-slicer[standalone]==1.11.0" "idc-index>=0.12.2" dcmqi

> If a later cell fails with an import or version error right after installation, use **Runtime → Restart session**, then run the notebook again **skipping the install cell above**. This is occasionally needed because the install upgrades packages already loaded by Colab.

## 2. Find a CT scan and its segmentation in IDC

We use `idc-index` to join the main `index` (one row per DICOM series) with `seg_index` (one row per DICOM Segmentation), matching each segmentation to the CT series it was computed on via `segmented_SeriesInstanceUID`. Here we look for small low‑dose chest CTs from the **NLST** collection that have a segmentation.

In [ ]:
from idc_index import IDCClient

client = IDCClient()
print("IDC data version:", client.get_idc_version())

client.fetch_index("seg_index")
candidates = client.sql_query("""
    SELECT i.collection_id,
           i.PatientID,
           i.SeriesInstanceUID         AS ct_series,
           i.instanceCount             AS ct_slices,
           ROUND(i.series_size_MB, 1)  AS ct_MB,
           s.SeriesInstanceUID         AS seg_series,
           i.license_short_name        AS license
    FROM seg_index s
    JOIN index i ON s.segmented_SeriesInstanceUID = i.SeriesInstanceUID
    WHERE i.Modality = 'CT'
      AND i.collection_id = 'nlst'
      AND i.instanceCount BETWEEN 80 AND 200
    ORDER BY i.series_size_MB
    LIMIT 5
""")
candidates

For a reproducible demo we pin to one specific pair from this collection. The CT is a single low‑dose chest acquisition (~80 slices, ~42 MB); the segmentation is a whole‑body organ segmentation from the IDC analysis result [**TotalSegmentator-CT-Segmentations**](https://doi.org/10.5281/zenodo.8347011) (71 structures). Both are licensed CC BY 4.0.

In [ ]:
# Pinned series for this tutorial (feel free to swap in a pair from `candidates` above).
ct_series  = "1.2.840.113654.2.55.71041873368734986406977154644223539362"
seg_series = "1.2.276.0.7230010.3.1.3.313263360.84.1706324554.366745"

## 3. Download the data

We download each series into its own folder with a *flat* layout (`dirTemplate=""`) so the DICOM files are easy to list.

In [ ]:
import glob
from pathlib import Path

ct_dir, seg_dir = "data/ct", "data/seg"
client.download_from_selection(downloadDir=ct_dir,  seriesInstanceUID=[ct_series],  dirTemplate="")
client.download_from_selection(downloadDir=seg_dir, seriesInstanceUID=[seg_series], dirTemplate="")

ct_files = sorted(glob.glob(f"{ct_dir}/*.dcm"))
seg_dcm  = glob.glob(f"{seg_dir}/*.dcm")[0]
print(f"Downloaded {len(ct_files)} CT slices")
print(f"Segmentation object: {seg_dcm}")

## 4. A reusable trame-slicer viewer

The class below wraps the boilerplate from the `trame-slicer` examples into a small, reusable viewer:

- a `SlicerApp` (the MRML scene, view manager, IO, display and volume‑rendering helpers);
- `register_rca_factories(...)` to stream the server‑side renders to the browser (the *remote‑controlled area*);
- a `LayoutManager` set to the **Quad View** (Axial / Coronal / Sagittal / 3D);
- methods to `load_volume(...)` and `load_segmentation(...)`, plus a *VR shift* slider that brightens/darkens the volume rendering.

`show_in_colab(...)` starts the trame server in the background and embeds it in the cell. In Colab it routes the server port through `google.colab.kernel.proxyPort`; elsewhere (e.g. local Jupyter) it falls back to `localhost`.

In [ ]:
import os
os.environ["TRAME_IPYWIDGETS_DISABLE"] = "1"  # use a plain iframe, not the ipywidgets backend

import socket
from IPython.display import IFrame

from trame.app import TrameApp, get_server
from trame_vuetify.ui.vuetify3 import SinglePageLayout
from trame_vuetify.widgets.vuetify3 import VSlider

from trame_slicer.core import LayoutManager, SlicerApp
from trame_slicer.rca_view import register_rca_factories


class IDCSlicerViewer(TrameApp):
    """Minimal 4-up (Axial / Coronal / Sagittal / 3D) Slicer viewer for a notebook."""

    _instances = 0

    def __init__(self, title="IDC trame-slicer viewer"):
        IDCSlicerViewer._instances += 1
        # A uniquely named server per instance so multiple viewers can coexist in one notebook.
        super().__init__(server=get_server(f"idc_slicer_{IDCSlicerViewer._instances}"))

        self._title = title
        self._volume_node = None
        self._preset_name = None

        self._slicer_app = SlicerApp()
        register_rca_factories(self._slicer_app.view_manager, self.server)

        self._layout = LayoutManager(
            self._slicer_app.scene,
            self._slicer_app.view_manager,
            self.server,
        )
        self._layout.register_layout_dict(LayoutManager.default_grid_configuration())
        self._layout.set_layout("Quad View")

        self.state.change("vr_shift")(self._on_vr_shift)
        self._build_ui()

    # ---- data loading -------------------------------------------------------
    def load_volume(self, dicom_files, vr_preset="CT-Chest-Contrast-Enhanced",
                    show_volume_rendering=True):
        app = self._slicer_app
        nodes = app.io_manager.load_volumes([str(f) for f in dicom_files])
        if not nodes:
            raise RuntimeError("Could not load the volume from the provided DICOM files.")
        self._volume_node = nodes[0]

        # Fall back gracefully if the requested rendering preset is unavailable.
        presets = app.volume_rendering.preset_names()
        if vr_preset not in presets:
            vr_preset = next((p for p in presets if "Chest" in p),
                             presets[0] if presets else "")
        self._preset_name = vr_preset

        app.display_manager.show_volume(self._volume_node, vr_preset=vr_preset,
                                        do_reset_views=True)
        if not show_volume_rendering:
            vr_display = app.volume_rendering.get_vr_display_node(self._volume_node)
            if vr_display:
                vr_display.SetVisibility(False)

        # Configure the VR-shift slider range from the preset.
        if self._preset_name:
            lo, hi = app.volume_rendering.get_preset_vr_shift_range(self._preset_name)
            self.state.vr_shift_min, self.state.vr_shift_max = lo / 10, hi / 10
        else:
            self.state.vr_shift_min, self.state.vr_shift_max = -125.0, 125.0
        self.state.vr_shift = 0.0
        return self._volume_node

    def load_segmentation(self, segmentation_file, show_3d=True):
        app = self._slicer_app
        seg = app.io_manager.load_segmentation(str(segmentation_file))
        if seg is None:
            raise RuntimeError("Could not load the segmentation file.")
        app.segmentation_editor.set_active_segmentation(seg, self._volume_node)
        app.segmentation_editor.set_surface_representation_enabled(show_3d)
        app.display_manager.reset_views()
        return seg

    # ---- interactivity ------------------------------------------------------
    def _on_vr_shift(self, vr_shift=0.0, **kwargs):
        if not self._volume_node or not self._preset_name:
            return
        self._slicer_app.volume_rendering.set_absolute_vr_shift_from_preset(
            self._volume_node, self._preset_name, float(vr_shift)
        )

    # ---- UI -----------------------------------------------------------------
    def _build_ui(self):
        with SinglePageLayout(self.server) as self.ui:
            self.ui.root.theme = "dark"
            with self.ui.toolbar:
                self.ui.toolbar.clear()
                self.ui.title.set_text(self._title)
                VSlider(
                    v_model=("vr_shift", 0),
                    min=("vr_shift_min", -125),
                    max=("vr_shift_max", 125),
                    step=1, label="VR shift", hide_details=True, density="compact",
                    style="max-width: 320px; margin: auto 16px;",
                )
            with self.ui.content:
                self._layout.initialize_layout_grid(self.ui)


def _free_port():
    s = socket.socket()
    s.bind(("", 0))
    port = s.getsockname()[1]
    s.close()
    return port


async def show_in_colab(app, height=720):
    """Start the trame server and embed the viewer in the current cell."""
    port = _free_port()
    app.server.start(port=port, exec_mode="task",
                     open_browser=False, show_connection_info=False)
    await app.server.ready
    try:
        from google.colab.output import eval_js
        url = eval_js(f"google.colab.kernel.proxyPort({port})")
    except ModuleNotFoundError:
        url = f"http://localhost:{port}/"
    return IFrame(src=url, width="100%", height=height)

## 5. Demo 1 — explore the CT volume

Run the cell, wait a few seconds for the viewer to connect, and interact directly in the embedded view:

- **Left‑drag** in the 3D view to rotate; **scroll** to zoom.
- **Scroll** in any slice view to page through slices.
- Drag the **VR shift** slider in the toolbar to brighten or darken the volume rendering.

In [ ]:
viewer = IDCSlicerViewer(title="NLST low-dose chest CT")
viewer.load_volume(ct_files, vr_preset="CT-Chest-Contrast-Enhanced")
await show_in_colab(viewer)

## 6. Demo 2 — overlay the segmentation

A DICOM Segmentation object is not a scalar image, so we first convert it to a labelmap NRRD with `dcmqi`'s `segimage2itkimage`. The `--mergeSegments` flag writes all (non‑overlapping) structures into a single multi‑label volume — exactly what the viewer's `load_segmentation` expects.

In [ ]:
import subprocess

seg_out = "data/seg_nrrd"
Path(seg_out).mkdir(parents=True, exist_ok=True)
subprocess.run(
    ["segimage2itkimage",
     "--inputDICOM", seg_dcm,
     "--outputType", "nrrd",
     "--prefix", "segment",
     "--outputDirectory", seg_out,
     "--mergeSegments"],
    check=True,
)
seg_nrrd = sorted(glob.glob(f"{seg_out}/*.nrrd"))[0]
print("Merged labelmap:", seg_nrrd)

Now load the CT and the segmentation into a fresh viewer. We turn the CT volume rendering off so the colored organ surfaces are clearly visible in the 3D view, while the segmentation is also overlaid on the slice views.

In [ ]:
viewer2 = IDCSlicerViewer(title="NLST CT + TotalSegmentator")
viewer2.load_volume(ct_files, vr_preset="CT-Chest-Contrast-Enhanced",
                    show_volume_rendering=False)
viewer2.load_segmentation(seg_nrrd, show_3d=True)
await show_in_colab(viewer2)

## 7. Recap and next steps

You loaded a CT scan and a segmentation straight from IDC and explored them in an interactive, server‑rendered 3D Slicer viewer running entirely inside Colab — no desktop install, no `itkwidgets`.

From here you can:
- Swap in any IDC series UID (the same `load_volume` path also reads MR, PT, and other modalities, and a list of NRRD/NIfTI files).
- Load expert or AI segmentations from other IDC analysis results the same way.
- Build a richer UI: trame-slicer also ships ready‑made `MedicalViewerApp` and `SegmentationApp` (segment editing) applications under `trame_slicer.app`.

**Attribution.** IDC data carries per‑collection licenses; always check them and cite the original sources. The cell below generates citations for the data used here.

In [ ]:
for citation in client.citations_from_selection(seriesInstanceUID=[ct_series, seg_series]):
    print(citation, "\n")

## Support

If you have questions about IDC or this tutorial:
- Ask on the [IDC user forum](https://discourse.canceridc.dev/).
- Browse the documentation at [learn.canceridc.dev](https://learn.canceridc.dev/).
- For trame-slicer itself, see the [trame-slicer repository](https://github.com/KitwareMedical/trame-slicer).

You can find more tutorials in the IDC-Tutorials repository: https://github.com/ImagingDataCommons/IDC-Tutorials

## Acknowledgments

Imaging Data Commons is a node within the broader NCI [Cancer Research Data Commons (CRDC)](https://datacommons.cancer.gov/) and has been funded in whole or in part with Federal funds from the NCI, NIH, under task order no. HHSN26110071 under contract no. HHSN261201500003l.

This notebook uses [`trame-slicer`](https://github.com/KitwareMedical/trame-slicer) by Kitware, built on [3D Slicer](https://www.slicer.org) and [trame](https://kitware.github.io/trame/). The segmentation shown is from the IDC analysis result *TotalSegmentator-CT-Segmentations* ([doi:10.5281/zenodo.8347011](https://doi.org/10.5281/zenodo.8347011)), computed on the NLST collection.

If you use IDC in your work, please cite:

> Fedorov, A., et al. "National Cancer Institute Imaging Data Commons: Toward Transparency, Reproducibility, and Scalability in Imaging Artificial Intelligence." *RadioGraphics* 43.12 (2023). https://doi.org/10.1148/rg.230180